**Without parallelization**

In [28]:
import time

def work(x):
    time.sleep(2)
    return x


if __name__ == '__main__':
    numbers = [1, 2, 3, 4]

    result = []

    start_time = time.time()
    for n in numbers:
        result.append(work(n))

    print(time.time()-start_time)

8.001784086227417


**With Parallelization**

In [29]:
from multiprocessing import Pool
import time

def work(x):
    time.sleep(2)
    return x

if __name__ == "__main__":

    numbers = [1, 2, 3, 4]

    start_time = time.time()
    with Pool() as pool:
        result = pool.map(work, numbers)

    print(time.time()-start_time)

4.033000230789185


**Other Stuffs May Needed To develop Parallel Algorithms**

In [30]:
from multiprocessing import Lock, Array
import ctypes
from itertools import accumulate

if __name__ == "__main__":

    # Shared array
    arr = Array(ctypes.c_int, [10,20,30])

    # Lock
    lock = Lock()

    with lock:
        arr[0] += 5

    print(list(arr))

    # Prefix sum
    degree = [2,4,3]

    prefix = [0]
    prefix.extend(accumulate(degree[:-1]))

    print(prefix)

[15, 20, 30]
[0, 2, 6]


Parallel BFS - 210042112

Setup and graph


In [36]:
import ctypes, time
import numpy as np
from multiprocessing import get_context, cpu_count

# fork = workers inherit the shared arrays, nothing gets copied
CTX = get_context("fork")
N_WORKERS = max(2, cpu_count())

def build_graph(n, deg, seed=1):
    rng = np.random.default_rng(seed)
    src = np.repeat(np.arange(n, dtype=np.int64), deg)
    dst = rng.integers(0, n, size=n * deg, dtype=np.int64)

    back = np.arange(n - 1, dtype=np.int64)          # path keeps the graph connected
    u = np.concatenate([src, dst, back, back + 1])
    v = np.concatenate([dst, src, back + 1, back])   # both directions
    keep = u != v
    u, v = u[keep], v[keep]

    # CSR: sort edges by source, then Offsets[v]..Offsets[v+1] is v's neighbour block
    order = np.argsort(u, kind="stable")
    edges = v[order].astype(np.int32)
    offsets = np.zeros(n + 1, dtype=np.int64)
    np.cumsum(np.bincount(u, minlength=n), out=offsets[1:])
    return offsets, edges

OFFSETS, EDGES = build_graph(300000, 10)
SOURCE = 0
print("n =", OFFSETS.size - 1, " m =", EDGES.size)

n = 300000  m = 6599966


**Without parallelization**

In [37]:
def python_bfs(offsets, edges, source):
    n = offsets.size - 1
    level = [-1] * n
    level[source] = 0
    frontier = [source]
    d = 0

    off, edg = offsets.tolist(), edges.tolist()      # lists are faster to index than numpy scalars
    while frontier:
        nxt = []
        for v in frontier:                           # one vertex at a time
            for k in range(off[v], off[v + 1]):
                w = edg[k]
                if level[w] == -1:
                    level[w] = d + 1
                    nxt.append(w)
        frontier = nxt
        d += 1
    return np.array(level, dtype=np.int32)

start_time = time.time()
level_py = python_bfs(OFFSETS, EDGES, SOURCE)
py_time = time.time() - start_time
print(py_time)

1.7704241275787354


**With parallelization**

Each worker takes a slice of the frontier and collects its edges in one array operation instead of
a Python loop. Nothing is sent to the workers except two integers per round, because the graph and
the level array live in shared memory. There is no lock: workers only read `level`, and the parent
alone writes it once the round ends.

In [38]:
def gather_neighbours(offsets, edges, frontier):
    """All neighbours of all frontier vertices, as one flat array."""
    starts = offsets[frontier]
    counts = offsets[frontier + 1] - starts          # degree of each frontier vertex
    total = int(counts.sum())
    if total == 0:
        return np.empty(0, dtype=np.int32)

    # position inside each vertex's own block: 0,1,2.. then restart for the next vertex
    within = np.arange(total, dtype=np.int64) - np.repeat(np.cumsum(counts) - counts, counts)
    return edges[np.repeat(starts, counts) + within]


G_OFFSETS = G_EDGES = G_LEVEL = G_FRONTIER = None

def init_worker(offsets, edges, level, frontier):
    # runs once per worker, when the Pool starts
    global G_OFFSETS, G_EDGES, G_LEVEL, G_FRONTIER
    G_OFFSETS, G_EDGES, G_LEVEL, G_FRONTIER = offsets, edges, level, frontier


def to_shared(arr, ctype):
    """Copy a numpy array into shared memory that every worker can read."""
    raw = CTX.RawArray(ctype, arr.size)
    out = np.frombuffer(raw, dtype=arr.dtype)
    out[:] = arr
    return out


def expand(bounds):
    s, e = bounds                                    # only these two ints are sent over
    front = G_FRONTIER[s:e].astype(np.int64)
    ngh = gather_neighbours(G_OFFSETS, G_EDGES, front)
    return np.unique(ngh[G_LEVEL[ngh] == -1])        # unique first, so less comes back


def parallel_bfs(offsets, edges, source, n_workers=N_WORKERS, cutoff=20000):
    n = offsets.size - 1
    sh_off = to_shared(offsets, ctypes.c_int64)
    sh_edg = to_shared(edges, ctypes.c_int32)
    sh_lvl = to_shared(np.full(n, -1, dtype=np.int32), ctypes.c_int32)
    sh_fro = to_shared(np.zeros(n, dtype=np.int32), ctypes.c_int32)

    sh_lvl[source] = 0
    sh_fro[0] = source
    fsize, d = 1, 0

    with CTX.Pool(n_workers, initializer=init_worker,
                  initargs=(sh_off, sh_edg, sh_lvl, sh_fro)) as pool:
        while fsize:
            if fsize < cutoff:
                # tiny frontier: splitting it would cost more than it saves
                front = sh_fro[:fsize].astype(np.int64)
                ngh = gather_neighbours(sh_off, sh_edg, front)
                nxt = np.unique(ngh[sh_lvl[ngh] == -1])
            else:
                step = -(-fsize // n_workers)        # one contiguous slice per worker
                tasks = [(s, min(s + step, fsize)) for s in range(0, fsize, step)]
                parts = pool.map(expand, tasks)
                nxt = np.unique(np.concatenate(parts))   # merge, dropping cross-worker duplicates
                nxt = nxt[sh_lvl[nxt] == -1]             # recheck: level may have moved on

            fsize = nxt.size
            if fsize:
                d += 1
                sh_lvl[nxt] = d                      # only the parent writes, so no race
                sh_fro[:fsize] = nxt
    return np.array(sh_lvl)


if __name__ == "__main__":
    start_time = time.time()
    level_par = parallel_bfs(OFFSETS, EDGES, SOURCE)
    par_time = time.time() - start_time
    print(par_time)

0.47773218154907227


Check

In [39]:
# levels are a property of the graph, so both versions must agree exactly
print("parallel matches python:", np.array_equal(level_py, level_par))
print("deepest level:", int(level_py.max()), " workers:", N_WORKERS)

print("\nwithout parallelization:", round(py_time, 3), "s")
print("with parallelization   :", round(par_time, 3), "s")
print("speedup                :", round(py_time / par_time, 2), "x")

parallel matches python: True
deepest level: 6  workers: 2

without parallelization: 1.77 s
with parallelization   : 0.478 s
speedup                : 3.71 x
